In [ ]:
import os
import shutil
import marimo as mo

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from tqdm import tqdm

from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

from monai.transforms import (
    Compose,
    RandFlipd,
    RandRotate90d,
    RandAffined,
    RandGaussianNoised,
    RandAdjustContrastd,
    RandGaussianSmoothd,
    RandFlipd,
    Rand3DElasticd,
    RandScaleIntensityd,
)
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import DynUNet
from monai.networks.utils import one_hot
from monai.inferers import sliding_window_inference

import wandb

##Config

In [ ]:
NUM_LABELS = 3
PATCH_SIZE = (96, 96, 96)
LR = 1e-4
BATCH_SIZE = 8
EPOCHS = 300

DATASET_REPO = "AG2307/pengwin-2026-fracture-segmentation"
HF_REPO = "AG2307/pengwin2026-anatomy-segmentation-swin"

CHECKPOINT_DIR = "/marimo/checkpoints"
LOCAL_CHECKPOINT = f"{CHECKPOINT_DIR}/best_model_swin_fdm.pth"

# torch.backends.cudnn.benchmark = True

##Build Datasets, Augmentations and Dataloaders

In [ ]:
class PelvicDataset(Dataset):
    def __init__(self, hf_dataset, transforms=None):
        self.dataset = hf_dataset
        self.transforms = transforms

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        if self.transforms:
            sample = self.transforms(sample)
        return sample


def build_transforms():
    return Compose(
        [
            # mirror flipping along all 3 axes
            RandFlipd(
                keys=["image", "gt_label", "fdm_weights"],
                prob=0.5,
                spatial_axis=0,
            ),
            RandFlipd(
                keys=["image", "gt_label", "fdm_weights"],
                prob=0.5,
                spatial_axis=1,
            ),
            RandFlipd(
                keys=["image", "gt_label", "fdm_weights"],
                prob=0.5,
                spatial_axis=2,
            ),
            # elastic deformation — the big one you're missing
            Rand3DElasticd(
                keys=["image", "gt_label", "fdm_weights"],
                mode=("bilinear", "nearest", "bilinear"),
                prob=0.3,
                sigma_range=(5, 8),
                magnitude_range=(50, 150),
                spatial_size=PATCH_SIZE,
            ),
            # affine — rotation range bumped to match paper (30 degrees ≈ 0.52 rad)
            RandAffined(
                keys=["image", "gt_label", "fdm_weights"],
                mode=("bilinear", "nearest", "bilinear"),
                prob=0.3,
                rotate_range=(0.52, 0.52, 0.52),
                scale_range=(0.2, 0.2, 0.2),
                translate_range=(20, 20, 20),
                padding_mode="border",
            ),
            # intensity augmentations (image only)
            RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.1),
            RandGaussianSmoothd(keys=["image"], prob=0.15, sigma_x=(0.5, 1.0)),
            RandAdjustContrastd(keys=["image"], prob=0.15, gamma=(0.75, 1.25)),
            RandScaleIntensityd(keys=["image"], prob=0.15, factors=0.25),
        ]
    )


def build_dataloaders():
    data = load_dataset(DATASET_REPO)
    data = data.remove_columns(
        [
            "bone",
            "case_id",
            "num_fragments",
            "original_spacing",
            "original_shape",
            "original_affine",
        ]
    )
    data = data.with_format("torch")

    train_dataset = PelvicDataset(data["train"], transforms=build_transforms())
    val_dataset = PelvicDataset(data["validation"], transforms=None)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        pin_memory=True,
    )

    return train_loader, val_loader

##Checkpointing Helpers

In [ ]:
def maybe_download_checkpoint(api, hf_token):
    if os.path.exists(LOCAL_CHECKPOINT):
        os.remove(LOCAL_CHECKPOINT)
        print("Removed stale local checkpoint.")

    try:
        downloaded = hf_hub_download(
            repo_id=HF_REPO,
            filename="latest_model_fdm.pth",
            repo_type="model",
            token=hf_token,
        )
        shutil.copy(downloaded, LOCAL_CHECKPOINT)
        print("Downloaded checkpoint from HuggingFace.")
    except Exception as e:
        print(f"No remote checkpoint found: {e}")


def load_checkpoint(model, optimizer, scheduler):
    if not os.path.exists(LOCAL_CHECKPOINT):
        print("No checkpoint found, starting fresh...")
        return 1, 0.0, None, 0

    print("Found existing checkpoint, loading...")
    ckpt = torch.load(LOCAL_CHECKPOINT, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_dice = ckpt["best_dice"]
    wandb_run_id = ckpt.get("wandb_run_id", None)
    global_step = ckpt.get("global_step", 0)
    print(f"  Resuming from epoch {start_epoch}, best Dice: {best_dice:.4f}")
    return start_epoch, best_dice, wandb_run_id, global_step


def save_checkpoint(
    model,
    optimizer,
    scheduler,
    global_step,
    epoch,
    best_dice,
    api,
    hf_token,
):
    raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": raw_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_dice": best_dice,
            "global_step": global_step,
            "wandb_run_id": wandb.run.id,
        },
        LOCAL_CHECKPOINT,
    )

##Weighted Dice + CE Loss Function

In [ ]:
class FDMWeightedDiceCELoss(nn.Module):
    def __init__(
        self,
        num_classes: int = 3,
        lambda_dice: float = 1.0,
        lambda_ce: float = 1.0,
        smooth: float = 1e-5,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.lambda_dice = lambda_dice
        self.lambda_ce = lambda_ce
        self.smooth = smooth

    def weighted_dice(
        self, pred: torch.Tensor, target: torch.Tensor, weights: torch.Tensor
    ) -> torch.Tensor:
        dice_loss = 0.0
        for c in range(self.num_classes):
            p = pred[:, c]
            t = target[:, c]
            w = weights.squeeze(1)

            num = 2.0 * (w * p * t).sum(dim=(1, 2, 3))
            den = (w * p).sum(dim=(1, 2, 3)) + (w * t).sum(dim=(1, 2, 3))
            dice_loss += 1.0 - (num + self.smooth) / (den + self.smooth)

        return (dice_loss / self.num_classes).mean()

    def weighted_ce(
        self, pred: torch.Tensor, target: torch.Tensor, weights: torch.Tensor
    ) -> torch.Tensor:
        target_squeezed = target.squeeze(1).long()
        w = weights.squeeze(1)

        ce = F.cross_entropy(pred, target_squeezed, reduction="none")

        weighted_ce = (w * ce).sum(dim=(1, 2, 3)) / w.sum(dim=(1, 2, 3))
        return weighted_ce.mean()

    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        fdm_weights: torch.Tensor,
    ) -> torch.Tensor:
        pred_soft = F.softmax(pred, dim=1)

        target_one_hot = (
            F.one_hot(target.squeeze(1).long(), self.num_classes)
            .permute(0, 4, 1, 2, 3)
            .float()
        )

        dice = self.weighted_dice(pred_soft, target_one_hot, fdm_weights)
        ce = self.weighted_ce(pred, target, fdm_weights)

        return self.lambda_dice * dice + self.lambda_ce * ce


class SmoothTransition:
    def __init__(self, tau_begin: int, tau_smooth: int, eps: float = 1e-6):
        self.tau_begin = tau_begin
        self.tau_smooth = tau_smooth
        self.eps = eps

    def get_weights(
        self, fdm_weights: torch.Tensor, step: int
    ) -> torch.Tensor:
        if step < self.tau_begin:
            return torch.ones_like(fdm_weights)

        if step > self.tau_begin + self.tau_smooth:
            return fdm_weights

        t = step - self.tau_begin
        delta = -torch.log(
            torch.tensor(1.0 - t / (self.tau_smooth + self.eps))
        ).item()

        uniform = torch.ones_like(fdm_weights)
        blended = (1.0 / (1.0 + delta)) * uniform + (
            delta / (1.0 + delta)
        ) * fdm_weights

        return blended

##Train and Eval Loop

In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    loss_fn,
    smooth_transition,
    global_step,
    device,
):
    model.train()
    total_loss = 0.0
    num_batches = len(loader)

    pbar = tqdm(loader, desc="Training")
    for i, batch in enumerate(pbar):
        image = batch["image"].to(device)
        label = batch["gt_label"].to(device)
        fdm_weights = batch["fdm_weights"].to(device)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            pred = model(image)
            weights = smooth_transition.get_weights(fdm_weights, global_step)
            outputs = torch.unbind(pred, dim=1)

            loss = loss_fn(outputs[0], label, weights)
            for j, aux_pred in enumerate(outputs[1:], 1):
                loss += (0.5**j) * loss_fn(aux_pred, label, weights)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        global_step += 1

        if i % 5 == 0:
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                avg_loss=f"{total_loss / (i + 1):.4f}",
            )

    return total_loss / num_batches, global_step


def evaluate(model, loader, loss_fn, device):
    dice_metric = DiceMetric(include_background=False, reduction="mean_batch")
    model.eval()

    total_loss = 0.0
    num_batches = len(loader)

    pbar = tqdm(loader, desc="Validation")
    for i, batch in enumerate(pbar):
        image = batch["image"].to(device)
        label = batch["gt_label"].to(device)
        fdm_weights = batch["fdm_weights"].to(device)

        with (
            torch.no_grad(),
            torch.autocast(device_type="cuda", dtype=torch.bfloat16),
        ):
            pred = sliding_window_inference(
                inputs=image,
                roi_size=PATCH_SIZE,
                sw_batch_size=16,
                predictor=model,
                overlap=0.5,
                mode="gaussian",
            )

            total_loss += loss_fn(pred, label, fdm_weights).item()

            pred_hard = torch.argmax(pred, dim=1, keepdim=True)
            dice_metric(
                one_hot(pred_hard, num_classes=NUM_LABELS),
                one_hot(label, num_classes=NUM_LABELS),
            )

        if i % 5 == 0:
            pbar.set_postfix(loss=f"{total_loss / (i + 1):.4f}")

    val_loss = total_loss / num_batches
    per_class = dice_metric.aggregate()
    val_dice = per_class.mean().item()
    dice_metric.reset()
    return val_loss, val_dice, per_class

##Model, Optimizer, Scheduler Definitions

In [ ]:
def build_model_and_optim(num_training_steps):
    kernel_size = [[3, 3, 3]] * 5
    strides = [[1, 1, 1]] + [[2, 2, 2]] * 4
    upsample_kernel_size = [[2, 2, 2]] * 4

    model = DynUNet(
        spatial_dims=3,
        in_channels=1,
        out_channels=NUM_LABELS,
        kernel_size=kernel_size,
        strides=strides,
        upsample_kernel_size=upsample_kernel_size,
        deep_supervision=True,
        deep_supr_num=3,
        res_block=True,
    )

    loss_fn = FDMWeightedDiceCELoss(num_classes=NUM_LABELS)
    smooth_transition = SmoothTransition(
        tau_begin=num_training_steps // 10,
        tau_smooth=num_training_steps // 5,
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.PolynomialLR(
        optimizer, total_iters=EPOCHS, power=0.9
    )

    return model, loss_fn, smooth_transition, optimizer, scheduler

##Main Loop

In [ ]:
def main():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    hf_token = os.environ.get("HF_TOKEN")
    device = "cuda" if torch.cuda.is_available() else "cpu"

    api = HfApi()
    api.create_repo(
        repo_id=HF_REPO,
        repo_type="model",
        private=True,
        token=hf_token,
        exist_ok=True,
    )

    train_loader, val_loader = build_dataloaders()
    num_training_steps = len(train_loader) * EPOCHS

    model, loss_fn, smooth_transition, optimizer, scheduler = (
        build_model_and_optim(num_training_steps)
    )
    model = model.to(device)

    maybe_download_checkpoint(api, hf_token)
    start_epoch, best_dice, wandb_run_id, global_step = load_checkpoint(
        model, optimizer, scheduler
    )

    model = torch.compile(model)

    wandb.init(
        project="pengwin2026-fdm-swin",
        id=wandb_run_id,
        resume="allow" if wandb_run_id else None,
        config={
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "patch_size": PATCH_SIZE[0],
            "num_classes": NUM_LABELS,
        },
    )

    CLASS_NAMES = ["main", "minor"]

    for epoch in range(start_epoch, EPOCHS + 1):
        print(f"Epoch {epoch}")
        train_loss, global_step = train_one_epoch(
            model,
            train_loader,
            optimizer,
            loss_fn,
            smooth_transition,
            global_step,
            device,
        )
        val_loss, val_dice, per_class = evaluate(
            model, val_loader, loss_fn, device
        )
        scheduler.step()

        class_str = " | ".join(
            f"{name}: {per_class[i]:.4f}" for i, name in enumerate(CLASS_NAMES)
        )
        print(
            f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}"
        )
        print(f"  Per-class: {class_str}")

        log_dict = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_dice": val_dice,
            "lr": optimizer.param_groups[0]["lr"],
        }
        for i, name in enumerate(CLASS_NAMES):
            log_dict[f"dice_{name}"] = per_class[i].item()

        wandb.log(log_dict)

        save_checkpoint(
            model,
            optimizer,
            scheduler,
            global_step,
            epoch,
            best_dice,
            api,
            hf_token,
        )

        api.upload_file(
            path_or_fileobj=LOCAL_CHECKPOINT,
            path_in_repo="latest_model_fdm.pth",
            repo_id=HF_REPO,
            repo_type="model",
            token=hf_token,
            commit_message=f"Epoch {epoch} | Dice: {val_dice:.4f}",
        )
        print("  Pushed current model to HuggingFace.")

        if val_dice > best_dice:
            best_dice = val_dice
            print(f"  New best! Dice: {best_dice:.4f}")
            api.upload_file(
                path_or_fileobj=LOCAL_CHECKPOINT,
                path_in_repo="best_model_fdm.pth",
                repo_id=HF_REPO,
                repo_type="model",
                token=hf_token,
                commit_message=f"Epoch {epoch} | Dice: {best_dice:.4f}",
            )
            print("  Pushed Best model to HuggingFace.")

    wandb.finish()


if __name__ == "__main__":
    main()